In [1]:

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
     

Defaulting to user installation because normal site-packages is not writeable
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ----

In [2]:
import pandas as pd
data = pd.read_csv('spam_dataset.csv')
data.head()

,sentence,label
0,The weather is nice today.,0
1,The weather is nice today.,0
2,Don't forget to submit your assignment.,0
3,Can you send me the report by evening?,0
4,I finished reading the book you suggested.,0


In [3]:
import numpy as np
import string
import nltk
from nltk.corpus import stopwords
nltk.download('punkt')
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
from nltk import WordNetLemmatizer
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()
nltk.download('wordnet')
nltk.download('omw-1.4')
import pandas as pd
import re

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [4]:
def preprocessing_pipeline(text):
    text=text.lower()
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F" 
        "\U0001F300-\U0001F5FF" 
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "\U0001f926-\U0001f937"
        "\U00010000-\U0010ffff"
        "\u2640-\u2642"
        "\u2600-\u2B55"
        "\u200d"
        "\u23cf"
        "\u23e9"
        "\u231a"
        "\ufe0f" 
        "\u3030"
        "]+", re.UNICODE)
    text = emoji_pattern.sub(r'', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    filtered_tokens = [word for word in tokens if word not in stop_words]
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]
    preprocessed_text = ' '.join(lemmatized_tokens)
    return preprocessed_text
data['preprocessed_message'] = data['sentence'].apply(preprocessing_pipeline)
data.head()

,sentence,label,preprocessed_message
0,The weather is nice today.,0,weather nice today
1,The weather is nice today.,0,weather nice today
2,Don't forget to submit your assignment.,0,dont forget submit assignment
3,Can you send me the report by evening?,0,send report evening
4,I finished reading the book you suggested.,0,finished reading book suggested


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
     

In [9]:
XV = data['preprocessed_message']
y = data['label']

In [10]:
vector= CountVectorizer()
x = vector.fit_transform(XV)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2)
     

In [18]:
class TextDataset(Dataset):
    def __init__(self, x , y):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]
     

In [20]:
train_loader = DataLoader(TextDataset(X_train.toarray(), y_train.values), batch_size=2, shuffle=True)
test_loader = DataLoader(TextDataset(X_test.toarray(), y_test.values), batch_size=2)

In [14]:
class CNNModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        kernels=4
        kernel_size=3
        self.conv = nn.Conv1d(1, kernels, kernel_size)

        pool_size=2
        self.pool = nn.MaxPool1d(pool_size)

        # Calculate flattened size
        flattened_size = kernels * ((input_size - pool_size)//2)

        # Hidden layer
        hidden_neurons=8
        self.fc1 = nn.Linear(flattened_size, hidden_neurons)   # 8 neurons (you can change)

        # Output layer
        output_neurons=1
        self.fc2 = nn.Linear(hidden_neurons, output_neurons)

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.unsqueeze(1)              # (batch, 1, features)
        x = torch.relu(self.conv(x))
        x = self.pool(x)

        x = x.view(x.size(0), -1)       # Flatten

        x = torch.relu(self.fc1(x))     # Hidden layer + activation
        x = self.fc2(x)                 # Output layer

        return self.sigmoid(x)
     

In [15]:
model = CNNModel(input_size=x.shape[1])


In [16]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [21]:
for epoch in range(10):
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch).squeeze()
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.6436
Epoch 2, Loss: 0.4833
Epoch 3, Loss: 0.2334
Epoch 4, Loss: 0.0015
Epoch 5, Loss: 0.0019
Epoch 6, Loss: 0.0011
Epoch 7, Loss: 0.0014
Epoch 8, Loss: 0.0011
Epoch 9, Loss: 0.0003
Epoch 10, Loss: 0.0007


In [23]:
y_pred = []
y_actual = []

model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch).squeeze()
        preds = (output > 0.5).int()
        y_pred.extend(preds.tolist())
        y_actual.extend(y_batch.tolist())
        
y_actual = [int(y) for y in y_actual]
print(y_pred)


[0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1]


In [24]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

print("\nConfusion Matrix:")
print(confusion_matrix(y_actual, y_pred))

print("\nClassification Report:")
print(classification_report(y_actual, y_pred))

print("\nAccuracy Score:", accuracy_score(y_actual, y_pred))


Confusion Matrix:
[[12  0]
 [ 0  8]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       1.00      1.00      1.00         8

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20


Accuracy Score: 1.0
